Creating Spatial Datasets AI and Total using a loopup table for the coordinates

In [1]:
import duckdb
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.transform import from_origin


In [2]:
db = duckdb.read_parquet(r"data/input/piebro/changeset_data/year=*/month=*/*.parquet", hive_partitioning=1) 

In [3]:
countries = gpd.read_file(r"UN_Countries_Simplified.shp")

In [4]:
countries= countries[["iso3cd", "nam_en", "geometry"]]

In [5]:
#for c2/ natural earth its SOV_A3
#for countries its ADM0_ISO
#for un = iso3cd
COUNTRY_COL = "iso3cd" 

## Extracting and Downloading Country (year) based CSV
**With Lookup table** -- Aggregation directly in duckdb

In [20]:
# ── 2. Get distinct grid cells with edit volume ───────────────────────────────
grid = duckdb.sql("""
    SELECT
        mid_pos_x,
        mid_pos_y,
        SUM(edit_count) AS total_edits
    FROM db
    WHERE mid_pos_x IS NOT NULL
      AND mid_pos_y IS NOT NULL
    GROUP BY mid_pos_x, mid_pos_y
""").df()

print(f"Grid cells: {len(grid):,}")
print(f"mid_pos_x range: {grid.mid_pos_x.min()} – {grid.mid_pos_x.max()}")
print(f"mid_pos_y range: {grid.mid_pos_y.min()} – {grid.mid_pos_y.max()}")

Grid cells: 30,006
mid_pos_x range: -35 – 360
mid_pos_y range: -125 – 180


In [21]:
# ── 3. Try BOTH coordinate conversions and check match rate ──────────────────

def test_conversion(grid, countries, lon_formula, lat_formula, label):
    g = grid.copy()
    g["lon"] = lon_formula(g["mid_pos_x"])
    g["lat"] = lat_formula(g["mid_pos_y"])
    
    gdf = gpd.GeoDataFrame(
        g,
        geometry=gpd.points_from_xy(g["lon"], g["lat"]),
        crs="EPSG:4326"
    )
    
    joined = gpd.sjoin(gdf, countries[[COUNTRY_COL, "geometry"]],
                       how="left", predicate="within")
    
    matched      = joined[COUNTRY_COL].notna()
    matched_edits = joined.loc[matched, "total_edits"].sum()
    total_edits   = joined["total_edits"].sum()
    
    print(f"\n── {label} ──")
    print(f"  Cells matched:  {matched.sum():,} / {len(joined):,} ({matched.mean()*100:.1f}%)")
    print(f"  Edits matched:  {matched_edits:,.0f} / {total_edits:,.0f} ({matched_edits/total_edits*100:.1f}%)")
    print(f"  Sample lon/lat: {g[['lon','lat']].head(3).to_string()}")
    return joined

# Version 1 — current formula
j1 = test_conversion(grid, countries,
    lon_formula = lambda x: x - 180.0,
    lat_formula = lambda y: 90.0 - y,
    label="lon=x-180, lat=90-y"
)

# Version 2 — flipped lat
j2 = test_conversion(grid, countries,
    lon_formula = lambda x: x - 180.0,
    lat_formula = lambda y: y - 90.0,
    label="lon=x-180, lat=y-90"
)


── lon=x-180, lat=90-y ──
  Cells matched:  7,212 / 30,006 (24.0%)
  Edits matched:  2,473,221,926 / 18,448,564,436 (13.4%)
  Sample lon/lat:     lon   lat
0  -2.0 -53.0
1  17.0 -62.0
2   1.0 -53.0

── lon=x-180, lat=y-90 ──
  Cells matched:  15,840 / 30,006 (52.8%)
  Edits matched:  16,409,875,195 / 18,448,564,436 (88.9%)
  Sample lon/lat:     lon   lat
0  -2.0  53.0
1  17.0  62.0
2   1.0  53.0


In [22]:
# ── 4. Use whichever version had better match rate ────────────────────────────
# Replace j1 with j2 below if version 2 was better

best_join = j2   # ← j2 better conversion results

coord_lookup = (
    best_join[["mid_pos_x", "mid_pos_y", COUNTRY_COL]]
    .rename(columns={COUNTRY_COL: "country"})
    .drop_duplicates(subset=["mid_pos_x", "mid_pos_y"])
)
coord_lookup["country"] = coord_lookup["country"].fillna("Unknown")

In [14]:
#for the total of the 2020-2025 timeframe

duckdb.register("coord_lookup", coord_lookup)

df_country_ai = duckdb.sql("""
    SELECT
        c.country,
        COUNT(DISTINCT db.user_name)         AS Contributors,
        CAST(SUM(db.edit_count) AS BIGINT)   AS Edits,
        CAST(COUNT(*) AS INTEGER)            AS Changesets
    FROM db
    JOIN coord_lookup c
      ON db.mid_pos_x = c.mid_pos_x
     AND db.mid_pos_y = c.mid_pos_y
    WHERE db.year >= 2020 AND db.year <2026
      AND (
            db.created_by = 'Rapid'
         OR array_to_string(db.source, ',') ILIKE '%microsoft/BuildingFootprints%'
         OR array_to_string(db.source, ',') ILIKE '%mapwithai%'
         OR array_to_string(db.source, ',') ILIKE '%esri/Google_Africa_Buildings%'
         OR array_to_string(db.source, ',') ILIKE '%esri/Google_Open_Buildings%'
         OR array_to_string(hashtags, ',') ILIKE '%mapwithai%'
      )
    GROUP BY country
    ORDER BY Edits DESC
""").df()

#df_country_ai.to_csv("../data/ouput/v2/country_ai_agg_edits_20_25.csv")

In [15]:
df_country_ai.to_csv("../data/ouput/v2/country_ai_agg_edits_20_25.csv", index = False)

In [16]:
df_country_ai=pd.read_csv("../data/ouput/v2/country_ai_agg_edits_20_25.csv")

In [3]:
df_country_ai = pd.read_csv("../data/ouput/v2/country_ai_agg_edits_20_26.csv").drop(columns={"Unnamed: 0"})

In [17]:
#for the total of the 2020-2025 timeframe

df_country_total = duckdb.sql("""
    SELECT 
        c.country,
        COUNT(DISTINCT user_name) as Contributors,
        CAST(SUM(edit_count) as BIGINT) as Edits,
        CAST(COUNT(*) AS INTEGER) as Changesets
    FROM db
    JOIN coord_lookup c
      ON db.mid_pos_x = c.mid_pos_x
     AND db.mid_pos_y = c.mid_pos_y
    WHERE db.year >= 2020 AND db.year <2026
    GROUP BY country
    ORDER BY Edits DESC
    
""").df()


In [18]:
df_country_total.to_csv("../data/ouput/v2/country_total_agg_edits_20_25.csv", index = False)

In [5]:
df_country_total = pd.read_csv("../data/ouput/v2/country_total_agg_edits_20_25.csv")

**Additionaly seperated by years**

In [16]:
df_year_country = duckdb.sql("""
    SELECT 
        c.country,
        COUNT(DISTINCT user_name) as Contributors,
        CAST(SUM(edit_count) as BIGINT) as Edits,
        CAST(COUNT(*) AS INTEGER) as Changesets,
        year
    FROM db
    JOIN coord_lookup c
      ON db.mid_pos_x = c.mid_pos_x
     AND db.mid_pos_y = c.mid_pos_y
    WHERE db.year >= 2020
    GROUP BY country,year

    
""").df()


In [15]:
df_year_country_25 = duckdb.sql("""
    SELECT 
        c.country,
        COUNT(DISTINCT user_name) as Contributors,
        CAST(SUM(edit_count) as BIGINT) as Edits,
        CAST(COUNT(*) AS INTEGER) as Changesets,
        year
    FROM db
    JOIN coord_lookup c
      ON db.mid_pos_x = c.mid_pos_x
     AND db.mid_pos_y = c.mid_pos_y
    WHERE db.year >= 2020 AND db.year <2026
    GROUP BY country,year

    
""").df()
df_year_country_25.to_csv("../data/ouput/v2/year_country_total_edits_20_25.csv")

In [17]:
df_year_country.to_csv("../data/ouput/v2/year_country_total_edits_20_26.csv")

In [18]:
df_ai_country_year = duckdb.sql("""
    SELECT
        c.country,
        COUNT(DISTINCT db.user_name)         AS Contributors,
        CAST(SUM(db.edit_count) AS BIGINT)   AS Edits,
        CAST(COUNT(*) AS INTEGER)            AS Changesets,
        year
    FROM db
    JOIN coord_lookup c
      ON db.mid_pos_x = c.mid_pos_x
     AND db.mid_pos_y = c.mid_pos_y
    WHERE db.year >= 2020
      AND (
            db.created_by = 'Rapid'
         OR array_to_string(db.source, ',') ILIKE '%microsoft/BuildingFootprints%'
         OR array_to_string(db.source, ',') ILIKE '%mapwithai%'
         OR array_to_string(db.source, ',') ILIKE '%esri/Google_Africa_Buildings%'
         OR array_to_string(db.source, ',') ILIKE '%esri/Google_Open_Buildings%'
         OR array_to_string(hashtags, ',') ILIKE '%mapwithai%'
      )
    GROUP BY country, year
    ORDER BY Edits DESC
""").df()

In [16]:
df_ai_country_year_25 = duckdb.sql("""
    SELECT
        c.country,
        COUNT(DISTINCT db.user_name)         AS Contributors,
        CAST(SUM(db.edit_count) AS BIGINT)   AS Edits,
        CAST(COUNT(*) AS INTEGER)            AS Changesets,
        year
    FROM db
    JOIN coord_lookup c
      ON db.mid_pos_x = c.mid_pos_x
     AND db.mid_pos_y = c.mid_pos_y
    WHERE db.year >= 2020 AND db.year <2026
      AND (
            db.created_by = 'Rapid'
         OR array_to_string(db.source, ',') ILIKE '%microsoft/BuildingFootprints%'
         OR array_to_string(db.source, ',') ILIKE '%mapwithai%'
         OR array_to_string(db.source, ',') ILIKE '%esri/Google_Africa_Buildings%'
         OR array_to_string(db.source, ',') ILIKE '%esri/Google_Open_Buildings%'
         OR array_to_string(hashtags, ',') ILIKE '%mapwithai%'
      )
    GROUP BY country, year
    ORDER BY Edits DESC
""").df()

df_ai_country_year_25.to_csv("../data/ouput/v2/year_country_ai_edits_20_25.csv")

In [19]:
df_ai_country_year.to_csv("../data/ouput/v2/year_country_ai_edits_20_26.csv")

##### **Check** Unmatched edits

In [28]:
duckdb.sql("""
    SELECT COUNT(*) as unmatched_changesets,
           SUM(edit_count) as unmatched_edits
    FROM db
    WHERE year >= 2019
      AND (mid_pos_x NOT BETWEEN 0 AND 359
       OR  mid_pos_y NOT BETWEEN 0 AND 179)
""").df()

,unmatched_changesets,unmatched_edits
0,1412,203435.0


In [37]:
duckdb.sql("""
    SELECT COUNT(*) as unmatched,
           SUM(edit_count) as unmatched_edits
    FROM db
    LEFT JOIN coord_lookup c
        ON db.mid_pos_x = c.mid_pos_x
       AND db.mid_pos_y = c.mid_pos_y
    WHERE db.year = 2025
      AND c.mid_pos_x IS NULL
""").df()

,unmatched,unmatched_edits
0,227040,46204.0


In [28]:
105908626-105912350

-3724

some unmatched edits (around 3000 - 4000 potentially per month) - likely due to mismatched mid x or mid y coordinates

## Cleaning Up data - Aggregated

In [7]:
df_country_ai = df_country_ai.rename(columns={"Contributors":"ContributorsAI", "Edits":"EditsAI", "Changesets":"ChangesetsAI" })

In [9]:
df_total= df_country_total.merge(df_country_ai, on="country", how="left")

In [11]:
df_total.sum()

country           USAUnknownCANRUSDEUFRABRAINDCHNPOLGBRJPNIDNNGA...
Contributors                                                1870456
Edits                                                    8294008204
Changesets                                                 96869665
ContributorsAI                                              37242.0
EditsAI                                                 573466009.0
ChangesetsAI                                              3833661.0
dtype: object

In [15]:
df_total.loc[df_total["country"]=="Unknown"]

,country,Contributors,Edits,Changesets,ContributorsAI,EditsAI,ChangesetsAI
1,Unknown,226192,913661526,11040413,3690.0,39147678.0,211223.0


In [17]:
#Percentage of Edits made with AI in the total timeframe 
(573466009/ 8294008204) *100

6.914220421477654

In [25]:
# Percentage of Contributors who use AI
(37242/1865589) * 100

1.9962596263164074

In [23]:
# Percentage of Changesets associated to AI
(3833661/96869665)*100

3.9575454297276655

In [19]:
#Percentage of Edits outside of coutnry boundaries
(913661526/8294008204)*100

11.015922621819485

In [27]:
#edits outside of ocutnry boundaries associated to AI
(37912742/876082132)*100

4.327532843690048

In [21]:
#changesets outside of ocuntry boundary
(11040413/96869665)*100

11.397183008736533

In [27]:
df_total["edits_ai_share"] = (df_total["EditsAI"]/df_total["Edits"])*100
df_total["contributors_ai_share"] = (df_total["ContributorsAI"]/df_total["Contributors"])*100
df_total["changesets_ai_share"] = (df_total["ChangesetsAI"]/df_total["Changesets"])*100


In [29]:
country_list = df_total.loc[(df_total["edits_ai_share"]>=6) & (df_total["Changesets"]>10000)].sort_values(by="edits_ai_share", ascending=False).reset_index()

In [31]:
country_list

,index,country,Contributors,Edits,Changesets,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share
0,16,TZA,14532,108973259,581717,1858.0,49302268.0,137285.0,45.242538,12.785577,23.599964
1,72,BOL,3711,19168732,109711,56.0,8277156.0,14310.0,43.180509,1.509027,13.043359
2,33,VNM,9251,55523870,834943,223.0,22082096.0,171050.0,39.770455,2.410550,20.486428
3,7,IND,62915,214504589,2839395,890.0,82508828.0,824666.0,38.464831,1.414607,29.043722
4,58,NZL,4061,27016954,256933,103.0,8074238.0,8545.0,29.885819,2.536321,3.325770
5,134,ALB,2876,4347265,57198,41.0,915037.0,3335.0,21.048567,1.425591,5.830623
6,0,USA,129892,1066638493,12636196,3717.0,220967460.0,1627821.0,20.716247,2.861608,12.882208
7,94,ECU,8804,12250492,226441,112.0,1906951.0,7026.0,15.566322,1.272149,3.102795
8,118,LAO,3175,6212666,57572,74.0,931489.0,3613.0,14.993386,2.330709,6.275620
9,32,KEN,14683,56736265,393700,1282.0,8075809.0,49362.0,14.233945,8.731186,12.537973


In [31]:
country_list.sort_values(by = "country")

,index,country,Contributors,Edits,Changesets,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share
24,74,AFG,4580,18890298,109059,77,1314810,1891,6.960240,1.681223,1.733924
5,134,ALB,2876,4347265,57198,36,852321,3048,19.605913,1.251739,5.328858
11,46,BGD,18099,32410666,391859,286,4345632,28917,13.408031,1.580198,7.379440
1,72,BOL,3711,19168732,109711,54,8068671,14002,42.092878,1.455133,12.762622
18,167,BRN,775,509232,11667,14,51517,308,10.116607,1.806452,2.639925
22,112,BTN,2030,7006005,41601,83,521233,5951,7.439803,4.088670,14.304945
29,39,COL,18947,42058043,620334,187,2636198,8202,6.268000,0.986964,1.322191
13,132,CYP,2300,4421491,72377,38,567021,1786,12.824203,1.652174,2.467635
8,94,ECU,8804,12250492,226441,107,1899374,7012,15.504471,1.215357,3.096612
30,79,ETH,6314,18427227,102137,120,1135569,2744,6.162452,1.900538,2.686588


In [55]:
df_total.to_csv("../data/ouput/v2/countries_edits_ai_20_25.csv", index = False)

In [25]:
map_data = countries.merge(df_total, left_on=COUNTRY_COL, right_on="country", how="left")

In [37]:
map_data.to_file("../data/ouput/v2/spatial/countries_edits_ai_20_25.geojson")

In [14]:
map_data = gpd.read_file("../data/ouput/v2/spatial/countries_edits_ai_20_25.geojson")

In [15]:
map_data

,SOVEREIGNT,SOV_A3,ADM0_DIF,TYPE,TLC,ADMIN,ADM0_A3,GEOU_DIF,GEOUNIT,GU_A3,...,Contributors,Edits,Changesets,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share,geometry
0,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,45181.0,143041213.0,1758078.0,1119.0,4398277.0,17051.0,3.074832,2.476705,0.969866,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
1,Malaysia,MYS,0,Sovereign country,1,Malaysia,MYS,0,Malaysia,MYS,...,5572.0,19740351.0,354508.0,138.0,1181927.0,10966.0,5.987366,2.476669,3.093301,"MULTIPOLYGON (((117.70361 4.16341, 117.69711 4..."
2,Chile,CHL,0,Sovereign country,1,Chile,CHL,0,Chile,CHL,...,5674.0,38468831.0,251631.0,101.0,1847698.0,8811.0,4.803104,1.780049,3.501556,"MULTIPOLYGON (((-69.51009 -17.50659, -69.50611..."
3,Bolivia,BOL,0,Sovereign country,1,Bolivia,BOL,0,Bolivia,BOL,...,3662.0,19306261.0,108619.0,54.0,8155252.0,14112.0,42.241488,1.474604,12.992202,"MULTIPOLYGON (((-69.51009 -17.50659, -69.51009..."
4,Peru,PER,0,Sovereign country,1,Peru,PER,0,Peru,PER,...,15754.0,32883386.0,393846.0,89.0,801317.0,1292.0,2.436845,0.564936,0.328047,"MULTIPOLYGON (((-69.51009 -17.50659, -69.63832..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,Bahrain,BHR,0,Sovereign country,1,Bahrain,BHR,0,Bahrain,BHR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((50.55161 26.19424, 50.59474 26..."
205,Spratly Islands,PGA,0,Indeterminate,1,Spratly Islands,PGA,0,Spratly Islands,PGA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((115.3672 10.23749, 115.36598 1..."
206,Bajo Nuevo Bank (Petrel Is.),BJN,0,Indeterminate,1,Bajo Nuevo Bank (Petrel Is.),BJN,0,Bajo Nuevo Bank (Petrel Is.),BJN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-79.98929 15.79495, -79.98782 ..."
207,Serranilla Bank,SER,0,Indeterminate,1,Serranilla Bank,SER,0,Serranilla Bank,SER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-78.63707 15.86209, -78.64041 ..."


In [17]:
map_data.loc[map_data["ADM0_A3"]=="PNG"]

,SOVEREIGNT,SOV_A3,ADM0_DIF,TYPE,TLC,ADMIN,ADM0_A3,GEOU_DIF,GEOUNIT,GU_A3,...,Contributors,Edits,Changesets,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share,geometry
154,Papua New Guinea,PNG,0,Sovereign country,1,Papua New Guinea,PNG,0,Papua New Guinea,PNG,...,3490.0,7997625.0,71843.0,41.0,1007513.0,3655.0,12.597652,1.174785,5.087482,"MULTIPOLYGON (((140.97446 -2.60052, 140.98732 ..."


## Cleaning Up Data - Yearly

In [30]:
df_year_country = pd.read_csv("../data/ouput/v2/year_country_total_edits_20_26.csv").drop(columns={"Unnamed: 0"})

In [32]:
df_year_ai_country = pd.read_csv("../data/ouput/v2/year_country_ai_edits_20_26.csv").drop(columns={"Unnamed: 0"})

In [43]:
df_year_country = df_year_country_25.copy()
df_year_ai_country = df_ai_country_year_25.copy()

In [44]:
df_year_ai_country = df_year_ai_country.rename(columns={"Contributors":"ContributorsAI", "Edits":"EditsAI", "Changesets":"ChangesetsAI" })

In [45]:
df_total_year= df_year_country.merge(df_year_ai_country, on=("country", "year"), how="left")

In [46]:
df_total_year["edits_ai_share"] = (df_total_year["EditsAI"]/df_total_year["Edits"])*100
df_total_year["contributors_ai_share"] = (df_total_year["ContributorsAI"]/df_total_year["Contributors"])*100
df_total_year["changesets_ai_share"] = (df_total_year["ChangesetsAI"]/df_total_year["Changesets"])*100


In [47]:
df_total_year

,country,Contributors,Edits,Changesets,year,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share
0,Unknown,56196,160928261,1992625,2020,931.0,7123445.0,79232.0,4.426472,1.656702,3.976262
1,ARG,2084,8911868,66946,2020,27.0,767700.0,1655.0,8.614356,1.295585,2.472142
2,ZAF,4066,18675240,195491,2020,14.0,260922.0,756.0,1.397155,0.344319,0.386719
3,IDN,18008,45395178,534671,2020,23.0,11323.0,56.0,0.024943,0.127721,0.010474
4,ALB,458,890302,6987,2020,7.0,15927.0,168.0,1.788944,1.528384,2.404465
...,...,...,...,...,...,...,...,...,...,...,...
1003,BRN,154,40849,1235,2025,1.0,17356.0,53.0,42.488188,0.649351,4.291498
1004,MLI,348,500244,3423,2025,10.0,2738.0,21.0,0.547333,2.873563,0.613497
1005,GUY,77,440989,482,2025,3.0,2380.0,25.0,0.539696,3.896104,5.186722
1006,GAB,107,126533,890,2025,3.0,991.0,5.0,0.783195,2.803738,0.561798


In [48]:
df_total_year = df_total_year.fillna(0) 

In [49]:
df_total_year.to_csv(r"../data/ouput/v2/country_stats_yearly_total_20_25.csv")

In [12]:
df_total_year= pd.read_csv(r"../data/ouput/v2/country_stats_yearly_total_20_25.csv")

In [13]:
df_total_year

,Unnamed: 0,country,Contributors,Edits,Changesets,year,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share
0,0,Unknown,56196,160928261,1992625,2020,931.0,7123445.0,79232.0,4.426472,1.656702,3.976262
1,1,ARG,2084,8911868,66946,2020,27.0,767700.0,1655.0,8.614356,1.295585,2.472142
2,2,ZAF,4066,18675240,195491,2020,14.0,260922.0,756.0,1.397155,0.344319,0.386719
3,3,IDN,18008,45395178,534671,2020,23.0,11323.0,56.0,0.024943,0.127721,0.010474
4,4,ALB,458,890302,6987,2020,7.0,15927.0,168.0,1.788944,1.528384,2.404465
...,...,...,...,...,...,...,...,...,...,...,...,...
1003,1003,BRN,154,40849,1235,2025,1.0,17356.0,53.0,42.488188,0.649351,4.291498
1004,1004,MLI,348,500244,3423,2025,10.0,2738.0,21.0,0.547333,2.873563,0.613497
1005,1005,GUY,77,440989,482,2025,3.0,2380.0,25.0,0.539696,3.896104,5.186722
1006,1006,GAB,107,126533,890,2025,3.0,991.0,5.0,0.783195,2.803738,0.561798


In [39]:
df_total_year.to_csv(r"../data/ouput/v2/country_stats_yearly_total.csv")

In [50]:
map_data_year = countries.merge(df_total_year, left_on=COUNTRY_COL, right_on="country", how="left")

In [51]:
map_data_year

,SOVEREIGNT,SOV_A3,ADM0_DIF,TYPE,TLC,ADMIN,ADM0_A3,GEOU_DIF,GEOUNIT,GU_A3,...,Contributors,Edits,Changesets,year,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share
0,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,18008.0,45395178.0,534671.0,2020.0,23.0,11323.0,56.0,0.024943,0.127721,0.010474
1,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,3638.0,21022203.0,260373.0,2022.0,50.0,972135.0,2547.0,4.624325,1.374382,0.978212
2,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,4022.0,9940906.0,128771.0,2024.0,200.0,400590.0,1948.0,4.029713,4.972650,1.512763
3,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,17223.0,37912737.0,544490.0,2021.0,50.0,93373.0,229.0,0.246284,0.290309,0.042058
4,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,3144.0,13465431.0,133318.0,2023.0,61.0,2338933.0,6971.0,17.369908,1.940204,5.228851
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1059,Bahrain,BHR,0,Sovereign country,1,Bahrain,BHR,0,Bahrain,BHR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1060,Spratly Islands,PGA,0,Indeterminate,1,Spratly Islands,PGA,0,Spratly Islands,PGA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1061,Bajo Nuevo Bank (Petrel Is.),BJN,0,Indeterminate,1,Bajo Nuevo Bank (Petrel Is.),BJN,0,Bajo Nuevo Bank (Petrel Is.),BJN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1062,Serranilla Bank,SER,0,Indeterminate,1,Serranilla Bank,SER,0,Serranilla Bank,SER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
map_data_year.to_file("../data/ouput/v2/spatial/countries_edits_ai_yearly.geojson")

In [52]:
map_data_year.to_file("../data/ouput/v2/spatial/countries_edits_ai_yearly_20_25.geojson")

In [9]:
map_data_year = gpd.read_file("../data/ouput/v2/spatial/countries_edits_ai_yearly.geojson")

In [11]:
map_data_year

,SOVEREIGNT,SOV_A3,ADM0_DIF,TYPE,TLC,ADMIN,ADM0_A3,GEOU_DIF,GEOUNIT,GU_A3,...,Edits,Changesets,year,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share,geometry
0,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,21022203.0,260373.0,2022.0,50.0,972135.0,2547.0,4.624325,1.374382,0.978212,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
1,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,9940906.0,128771.0,2024.0,200.0,400590.0,1948.0,4.029713,4.972650,1.512763,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
2,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,45395178.0,534671.0,2020.0,23.0,11323.0,56.0,0.024943,0.127721,0.010474,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
3,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,37912737.0,544490.0,2021.0,50.0,93373.0,229.0,0.246284,0.290309,0.042058,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
4,Indonesia,IDN,0,Sovereign country,1,Indonesia,IDN,0,Indonesia,IDN,...,13465431.0,133318.0,2023.0,61.0,2338933.0,6971.0,17.369908,1.940204,5.228851,"MULTIPOLYGON (((117.70361 4.16341, 117.70361 4..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1230,Bahrain,BHR,0,Sovereign country,1,Bahrain,BHR,0,Bahrain,BHR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((50.55161 26.19424, 50.59474 26..."
1231,Spratly Islands,PGA,0,Indeterminate,1,Spratly Islands,PGA,0,Spratly Islands,PGA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((115.3672 10.23749, 115.36598 1..."
1232,Bajo Nuevo Bank (Petrel Is.),BJN,0,Indeterminate,1,Bajo Nuevo Bank (Petrel Is.),BJN,0,Bajo Nuevo Bank (Petrel Is.),BJN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-79.98929 15.79495, -79.98782 ..."
1233,Serranilla Bank,SER,0,Indeterminate,1,Serranilla Bank,SER,0,Serranilla Bank,SER,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"MULTIPOLYGON (((-78.63707 15.86209, -78.64041 ..."


In [45]:
map_data_year.loc[map_data_year["ADM0_A3"]=="US1"]

,SOVEREIGNT,SOV_A3,ADM0_DIF,TYPE,TLC,ADMIN,ADM0_A3,GEOU_DIF,GEOUNIT,GU_A3,...,Edits,Changesets,year,ContributorsAI,EditsAI,ChangesetsAI,edits_ai_share,contributors_ai_share,changesets_ai_share,geometry
1006,United States of America,US1,0,Sovereignty,None,United States of America,US1,0,United States of America,US1,...,187084861.0,1626914.0,2022.0,585.0,44053822.0,135850.0,23.547508,2.007963,8.350165,"MULTIPOLYGON (((-122.75302 48.99251, -122.6532..."
1007,United States of America,US1,0,Sovereignty,None,United States of America,US1,0,United States of America,US1,...,168604804.0,1520764.0,2024.0,972.0,34308892.0,115473.0,20.348704,3.200843,7.593091,"MULTIPOLYGON (((-122.75302 48.99251, -122.6532..."
1008,United States of America,US1,0,Sovereignty,None,United States of America,US1,0,United States of America,US1,...,180618386.0,2801330.0,2021.0,1066.0,39226499.0,435408.0,21.717888,4.116625,15.542903,"MULTIPOLYGON (((-122.75302 48.99251, -122.6532..."
1009,United States of America,US1,0,Sovereignty,None,United States of America,US1,0,United States of America,US1,...,175095530.0,1550276.0,2023.0,862.0,33853290.0,116086.0,19.334183,2.942482,7.488086,"MULTIPOLYGON (((-122.75302 48.99251, -122.6532..."
1010,United States of America,US1,0,Sovereignty,None,United States of America,US1,0,United States of America,US1,...,169867088.0,1803458.0,2025.0,1100.0,32879773.0,155093.0,19.356176,2.807125,8.599757,"MULTIPOLYGON (((-122.75302 48.99251, -122.6532..."
1011,United States of America,US1,0,Sovereignty,None,United States of America,US1,0,United States of America,US1,...,43455175.0,499929.0,2026.0,503.0,10243935.0,73090.0,23.573567,2.622113,14.620076,"MULTIPOLYGON (((-122.75302 48.99251, -122.6532..."
1012,United States of America,US1,0,Sovereignty,None,United States of America,US1,0,United States of America,US1,...,193807642.0,3433912.0,2020.0,860.0,26401249.0,596821.0,13.622398,3.269962,17.380207,"MULTIPOLYGON (((-122.75302 48.99251, -122.6532..."
